# Modeling

It is the main nootebook related to building prediction models for https://www.kaggle.com/datasets/sahistapatel96/bankadditionalfullcsv dataset.

Previous notbook related to data preprocessing and feature engineering - https://github.com/Maxstef/ml-bank-additional-project/blob/main/notebooks/02_preprocessing.ipynb


## Notebook initialization

In [1]:
# Notebook initialization
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd()

while not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# print("Project root:", ROOT)

## Imports & Load data

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier


from src.data import load_raw_data

raw_df = load_raw_data()
raw_df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
target_col = "y"

## Split Data Train & Validation

In [4]:
from src.data import split_train_val, split_X_y

train_df, val_df = split_train_val(raw_df, stratify_col=target_col)

X_train, X_val, y_train, y_val = split_X_y(train_df, val_df, target_col)

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((32950, 20), (8238, 20), (32950,), (8238,))

## Experiments Preparation & Results Tracking

In this section we set up a simple framework for running and tracking modeling experiments.

We reuse the `build_pipeline` function implemented in the [previous notebook](https://github.com/Maxstef/ml-bank-additional-project/blob/main/notebooks/02_preprocessing.ipynb) to construct configurable preprocessing and modeling pipelines.

An empty `results_df` DataFrame is created to store the results of all experiments performed in this notebook.

To automate experiment tracking, we implement an `experiment_logger` decorator that logs model parameters, pipeline configuration, and evaluation metrics (F1 and AUROC for train/test).

The `train_pipeline` function builds and fits the pipeline. Since it is decorated with `experiment_logger`, each call automatically records the experiment results in `results_df`.

In [5]:
from src.pipelining import build_pipeline
# Global experiment table
results_df = pd.DataFrame()
experiment_counter = 0

In [6]:
import time
from sklearn.metrics import f1_score, roc_auc_score

def experiment_logger(func):

    def wrapper(*args, **kwargs):
        global results_df
        global experiment_counter

        start = time.time()

        # Run training function
        pipe, X_train, X_val, y_train, y_val, pipeline_params = func(*args, **kwargs)

        fit_time = time.time() - start

        # Predictions
        y_train_pred = pipe.predict(X_train)
        y_val_pred = pipe.predict(X_val)

        y_train_prob = pipe.predict_proba(X_train)[:, 1]
        y_val_prob = pipe.predict_proba(X_val)[:, 1]

        train_f1 = f1_score(y_train, y_train_pred, pos_label="yes")
        val_f1 = f1_score(y_val, y_val_pred, pos_label="yes")

        train_auc = roc_auc_score(y_train, y_train_prob)
        val_auc = roc_auc_score(y_val, y_val_prob)

        model = pipe.named_steps["classifier"]

        # ------------------------
        # model hyperparameters
        # ------------------------
        default_model = model.__class__()
        params = model.get_params()
        default_params = default_model.get_params()

        model_params = {
            f"model__{k}": v
            for k, v in params.items()
            if k in default_params and v != default_params[k]
        }

        # ------------------------
        # pipeline params
        # ------------------------
        pipe_params = {
            f"pipe__{k}": v
            for k, v in pipeline_params.items()
            if k != "model"
        }

        # keep features count
        n_features =len(pipe.named_steps["preprocessing"].get_feature_names_out())

        row = {
            "experiment_id": experiment_counter,
            "model_name": model.__class__.__name__,
            "fit_time": fit_time,
            "train_f1": train_f1,
            "val_f1": val_f1,
            "train_auroc": train_auc,
            "val_auroc": val_auc,
            "n_features": n_features,
        }

        row.update(pipe_params)
        row.update(model_params)

        results_df = pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

        experiment_counter += 1

        return pipe

    return wrapper

In [7]:
@experiment_logger
def train_pipeline(X_train, X_val, y_train, y_val, pipeline_params):

    pipe = build_pipeline(**pipeline_params)

    pipe.fit(X_train, y_train)

    return pipe, X_train, X_val, y_train, y_val, pipeline_params

## Base Models

Use LogisticRegression, KNeighborsClassifier and DecisionTreeClassifier as base models experiments

### Baseline models
Train few baseline models with default model and pipeline params

In [8]:
base_models = [
    LogisticRegression(solver="liblinear", random_state=42),
    KNeighborsClassifier(),
    DecisionTreeClassifier(max_leaf_nodes=100, random_state=42), # Limit max_leaf_nodes=100 to avoid overfitting
]

for base_model in base_models:
    train_pipeline(X_train, X_val, y_train, y_val,
    {
        "model": base_model
    }
)

In [9]:
results_df.sort_values("val_f1", ascending=False).head()

,experiment_id,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,n_features,model__random_state,model__solver,model__max_leaf_nodes
2,2,DecisionTreeClassifier,0.162604,0.452679,0.407924,0.795160,0.790662,59,42.0,NaN,100.0
1,1,KNeighborsClassifier,0.094648,0.488967,0.383562,0.926449,0.746241,59,NaN,NaN,NaN
0,0,LogisticRegression,0.173218,0.339418,0.337408,0.792315,0.800625,59,42.0,liblinear,NaN


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Conclusions from baseline models:**

- The baseline **DecisionTreeClassifier** achieves the best **validation F1 score**, indicating thebest performance on the classification objective among baseline models.
- The baseline **LogisticRegression** achieves the best **validation AUROC**, which is expected since logistic regression produces well-calibrated probability estimates.
- The **KNeighborsClassifier** shows the highest scores on the **training set** but noticeably worse performance on the **validation set**, suggesting a tendency to **overfit** the training data.

</span>

### Logistic Regression

#### Feature Statistical Significance

In this section, we explore the **statistical significance of features** for the baseline Logistic Regression model using `statsmodels.Logit`.

This allows us to:

- Estimate **coefficient significance** (p-values) for each feature  
- Identify which features **increase or decrease** the probability of the target outcome  
- Gain insights into the **most influential predictors** in our model  
- Verify whether the assumptions made in the **EDA step** were correct

Features with **p-values below 0.05** are considered statistically significant.

In [10]:
import statsmodels.api as sm

# STEP 1 - train pipeline avoiding any multicollinearity or other stuff that might spoil statistics
pipe_lr = train_pipeline(X_train, X_val, y_train, y_val,
    {
        "poly_degree": 1,                   # no poly features
        "drop_cols": None,                  # include all cols here (except of "duration")
        "pdays_transform_mode": "binary",   # "binary" to avoid multicollinearity and all p-values 1
        "drop_cats": {                      # drop at least one value per each category to avoid multicollinearity and all p-values 1
            "job": ["unknown"],
            "marital": ["unknown"],
            "education": ["unknown", "illiterate"],
            "default": ["no"],
            "housing": ["yes"],
            "loan": ["yes"],
            "contact": ["telephone"],
            "month": ["dec"],
            "day_of_week": ["fri"],
            "poutcome": ["nonexistent"],
        },
        "model": base_models[0],            # LogisticRegression(solver="liblinear", random_state=42)
    }
)

# STEP 2 - get transformet input data and feature names
X_train_transformed = pipe_lr[:-1].transform(X_train)
feature_names = pipe_lr.named_steps["preprocessing"].get_feature_names_out()
X_train_df = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=X_train.index
)

# STEP 3 - fit sm.Logit for calculating statistics
X_train_df_const = sm.add_constant(X_train_df)
logit_model = sm.Logit(y_train.map({"no":0, "yes":1}), X_train_df_const)
stats_result_lr = logit_model.fit()

# STEP 4 - print all statistics
print(stats_result_lr.summary())

Optimization terminated successfully.
         Current function value: 0.277029
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                32950
Model:                          Logit   Df Residuals:                    32899
Method:                           MLE   Df Model:                           50
Date:                Mon, 16 Mar 2026   Pseudo R-squ.:                  0.2131
Time:                        18:08:29   Log-Likelihood:                -9128.1
converged:                       True   LL-Null:                       -11599.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                         coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------
const                                 -1.6897      0.484    

In [11]:
# STEP 5 - print features sorted by significance (p-value) - round to 3 decimal and sort by raw p-value
significance_df = pd.DataFrame({
    "feature": stats_result_lr.params.index,
    "coef": stats_result_lr.params.values.round(3),
    "p_value": stats_result_lr.pvalues.values.round(3),
    "p_value_raw": stats_result_lr.pvalues.values,
})

significance_df = significance_df.sort_values("p_value_raw")

significance_df[["feature", "coef", "p_value"]]

,feature,coef,p_value
46,numeric__emp.var.rate,-2.391,0.000
26,cat__contact_cellular,0.751,0.000
47,numeric__cons.price.idx,1.285,0.000
30,cat__month_jun,-1.264,0.000
32,cat__month_may,-0.958,0.000
33,cat__month_nov,-0.956,0.000
31,cat__month_mar,0.974,0.000
48,numeric__cons.conf.idx,0.149,0.000
40,cat__poutcome_failure,-0.392,0.000
21,cat__default_risk,-0.238,0.000


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Feature Statistical Significance – Conclusions**

- Most **socio-economic context features** are statistically significant, except for `euribor3m`. This aligns with our EDA assumption that `euribor3m` is highly correlated with other socio-economic features and does not add new information to the model.  
- The features `previous` and `pdays` do not show statistical significance in the Logistic Regression model. This may be due to strong **class imbalance** (e.g., only ~4% of rows have `pdays != 999`). They might still be useful for **tree-based models**.  
- Numeric `age` does not appear significant as-is; using **binned age groups** may improve linear model performance, while tree-based models can handle raw numeric values.  
- Most **month** and **day_of_week** values are significant, suggesting **time-related effects** influence the target.  
- `housing` and `loan` remain insignificant, consistent with the EDA assumptions.  
- No statistically significant effects are observed for categorical variables `education`, `marital`, or `job` in the Logistic Regression model. This may be due to **high cardinality**.  
- Features such as `poutcome_failure`, `default_risk`, `contact_cellular`, and `campaign` are significant predictors of the target.

Overall, these results largely confirm the patterns observed in EDA and provide guidance for **feature selection and transformations** for linear models.
</span>

#### Drop Columns

Experiment with dropping additional columns for the Logistic Regression model based on the statistical significance analysis above.

In [12]:
# based on stats significans experiment above
drop_cols_options = [
    ["duration", "loan", "housing"],
    ["duration", "loan", "housing", "euribor3m"],
    ["duration", "loan", "housing", "euribor3m", "job"],
    ["duration", "loan", "housing", "euribor3m", "pdays"],
    ["duration", "loan", "housing", "euribor3m", "age"],
    ["duration", "loan", "housing", "euribor3m", "previous"],
    ["duration", "loan", "housing", "euribor3m", "job", "marital"],
    ["duration", "loan", "housing", "euribor3m", "job", "education", "marital"],
    ["duration", "loan", "housing", "euribor3m", "job", "education", "marital", "age"],
    ["duration", "loan", "housing", "euribor3m", "job", "education", "marital", "pdays"],
    ["duration", "loan", "housing", "euribor3m", "pdays", "age"],
    ["duration", "loan", "housing", "euribor3m", "pdays", "age", "previous"],
    ["duration", "loan", "housing", "euribor3m", "job", "pdays", "age", "previous"],
    ["duration", "loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous"],
    ["duration", "loan", "housing", "euribor3m", "job", "education", "marital", "pdays", "previous", "age"],
]

for drop_cols in drop_cols_options:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "drop_cols": drop_cols,
            "model": base_models[0],
        }
    )

In [13]:
def show_results_df(
    model_name="LogisticRegression",
    sort_by="val_f1",
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc"],
    show_count=10,
    max_experiment_id=np.inf,   # to reproduce output in case of re-run and do not show stored further experiment
):
    # Show full column content without truncation
    with pd.option_context("display.max_colwidth", None):
        display(
            results_df[
                (results_df["model_name"] == model_name) &
                (results_df["experiment_id"] <= max_experiment_id)
            ].sort_values(sort_by, ascending=False)[show_cols].head(show_count)
        )

show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__drop_cols", "n_features"],
    max_experiment_id=18
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__drop_cols,n_features
5,LogisticRegression,0.144977,0.341337,0.338762,0.792158,0.800931,"[duration, loan, housing, euribor3m]",52
8,LogisticRegression,0.147310,0.341600,0.338762,0.792140,0.800937,"[duration, loan, housing, euribor3m, age]",51
9,LogisticRegression,0.143457,0.338616,0.338487,0.792070,0.800971,"[duration, loan, housing, euribor3m, previous]",51
4,LogisticRegression,0.159340,0.339676,0.337959,0.792187,0.801095,"[duration, loan, housing]",53
0,LogisticRegression,0.173218,0.339418,0.337408,0.792315,0.800625,NaN,59
12,LogisticRegression,0.099081,0.338901,0.336894,0.790776,0.802232,"[duration, loan, housing, euribor3m, job, education, marital, age]",31
11,LogisticRegression,0.098267,0.339706,0.334975,0.790636,0.802216,"[duration, loan, housing, euribor3m, job, education, marital]",32
18,LogisticRegression,0.078731,0.334682,0.334975,0.790245,0.802289,"[duration, loan, housing, euribor3m, job, education, marital, pdays, previous, age]",26
6,LogisticRegression,0.127659,0.338976,0.334975,0.791390,0.800902,"[duration, loan, housing, euribor3m, job]",41
14,LogisticRegression,0.149205,0.333669,0.334694,0.791607,0.800880,"[duration, loan, housing, euribor3m, pdays, age]",47


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Feature Removal Experiments – Logistic Regression Conclusions**

- Removing `loan`, `housing`, and the leakage feature `duration` does **not affect model performance**, confirming their low predictive value.
- Excluding the correlated feature `euribor3m` also **does not degrade validation metrics**, suggesting redundancy with other macroeconomic variables.
- Removing additional variables (`age`, `previous`, `job`, `education`, `marital`) leads to **only minor changes in F1**, while **AUROC remains stable (~0.80)**.
- Even with aggressive feature reduction (59 → ~26 features), the model maintains **similar predictive performance** and **faster training time**.
- These results are **consistent with the earlier statistical significance exploration**, which showed that many of these features have limited impact on the Logistic Regression model.

**Summary**

Logistic Regression performance is **robust to feature removal**, confirming that a **smaller subset of features can achieve comparable results**, matching the insights from the statistical significance analysis.

</span>

#### Polynomial Degree

Create polynomial degree features for the Logistic Regression experiment.

In [14]:
poly_degree_options = [1,2,3,4]

for poly_degree in poly_degree_options:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "poly_degree": poly_degree,
            "drop_cols": ["duration", "loan", "housing", "euribor3m", "age"],
            "model": base_models[0],
        }
    )

In [15]:
show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__poly_degree", "n_features"],
    show_count=10,
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__poly_degree,n_features
22,LogisticRegression,90.472476,0.369552,0.341110,0.800306,0.800167,4.0,373
19,LogisticRegression,0.142201,0.341600,0.338762,0.792140,0.800937,1.0,51
5,LogisticRegression,0.144977,0.341337,0.338762,0.792158,0.800931,NaN,52
8,LogisticRegression,0.147310,0.341600,0.338762,0.792140,0.800937,NaN,51
9,LogisticRegression,0.143457,0.338616,0.338487,0.792070,0.800971,NaN,51
4,LogisticRegression,0.159340,0.339676,0.337959,0.792187,0.801095,NaN,53
20,LogisticRegression,0.557122,0.344979,0.337959,0.794696,0.801402,2.0,79
21,LogisticRegression,5.613992,0.358791,0.337408,0.797933,0.801355,3.0,163
0,LogisticRegression,0.173218,0.339418,0.337408,0.792315,0.800625,NaN,59
12,LogisticRegression,0.099081,0.338901,0.336894,0.790776,0.802232,NaN,31


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Polynomial Feature Experiments – Logistic Regression Conclusions**

- **Degree 1 (linear)** gives baseline performance: val F1 ~0.338–0.339, AUROC ~0.801, minimal training time.
- **Degree 2–4** slightly increase training F1 but **do not improve validation F1**, showing **overfitting**. Feature count and training time grow substantially.
- **Conclusion:** Linear features are sufficient; polynomial expansion adds complexity without improving validation performance

</span>

#### Age Binning

Experiment with grouping the numeric `age` feature into categorical groups.

In [16]:
age_bin_mode_options = [None, "group", "range"]

for age_bin_mode in age_bin_mode_options:
    train_pipeline(X_train, X_val, y_train, y_val,
            {
                "age_bin_mode": age_bin_mode,
                "drop_cols": ["duration", "loan", "housing", "euribor3m"],
                "model": base_models[0],
            }
        )

In [17]:
show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__age_bin_mode", "n_features"],
    show_count=6,
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__age_bin_mode,n_features
24,LogisticRegression,0.155966,0.342184,0.345219,0.792346,0.801223,group,54
22,LogisticRegression,90.472476,0.369552,0.341110,0.800306,0.800167,NaN,373
25,LogisticRegression,0.151042,0.340655,0.340909,0.792483,0.801066,range,57
19,LogisticRegression,0.142201,0.341600,0.338762,0.792140,0.800937,NaN,51
5,LogisticRegression,0.144977,0.341337,0.338762,0.792158,0.800931,NaN,52
23,LogisticRegression,0.151504,0.341337,0.338762,0.792158,0.800931,None,52


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Age Binning – Logistic Regression - Conclusions**

- Grouping the numeric `age` feature into **three categories (young / middle / old)** improves the model's performance.  
- Creating **more detailed age ranges** also improves performance, but the **three-category grouping appears to provide the best balance**.

</span>

#### Cyclical Encoding for Calendar Columns

Experiment with different encoding strategies for the `month` and `day_of_week` features.

In [18]:
calendar_cols_mode_options = ["onehot", "num", "cyclical"]

for calendar_cols_mode in calendar_cols_mode_options:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "calendar_cols_mode": calendar_cols_mode,
            "age_bin_mode": "group",
            "drop_cols": ["duration", "loan", "housing", "euribor3m"],
            "model": base_models[0],
        }
    )

In [19]:
show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__calendar_cols_mode", "n_features"],
    show_count=5,
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__calendar_cols_mode,n_features
24,LogisticRegression,0.155966,0.342184,0.345219,0.792346,0.801223,NaN,54
26,LogisticRegression,0.156620,0.342184,0.345219,0.792346,0.801223,onehot,54
22,LogisticRegression,90.472476,0.369552,0.341110,0.800306,0.800167,NaN,373
25,LogisticRegression,0.151042,0.340655,0.340909,0.792483,0.801066,NaN,57
8,LogisticRegression,0.147310,0.341600,0.338762,0.792140,0.800937,NaN,51


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Cyclical Encoding – Logistic Regression - Conclusions**

- Cyclical encoding of the `month` and `day_of_week` features does not improves the model's performance.
- One hot encoding look optimal option for `month` and `day_of_week` features in our dataset

</span>

#### Social and Economic Context Attributes

Experiment with binning socio-economic context features into categorical groups.

In [20]:
soceco_bin_cols_options = [
    ["emp.var.rate", "cons.price.idx", "cons.conf.idx", "nr.employed"],
    ["emp.var.rate", "cons.price.idx", "cons.conf.idx"],
    ["emp.var.rate", "cons.price.idx", "nr.employed"],
    ["emp.var.rate", "cons.conf.idx", "nr.employed"],
    ["cons.price.idx", "cons.conf.idx", "nr.employed"],
    ["emp.var.rate", "nr.employed"],
    ["cons.price.idx", "cons.conf.idx"],
]

for soceco_bin_cols in soceco_bin_cols_options:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "soceco_bin_cols": soceco_bin_cols,
            "calendar_cols_mode": "onehot",
            "age_bin_mode": "group",
            "drop_cols": ["duration", "loan", "housing", "euribor3m"],
            "model": base_models[0],
        }
    )

In [21]:
show_results_df(
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__soceco_bin_cols", "n_features"],
    show_count=10,
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__soceco_bin_cols,n_features
26,LogisticRegression,0.156620,0.342184,0.345219,0.792346,0.801223,NaN,54
24,LogisticRegression,0.155966,0.342184,0.345219,0.792346,0.801223,NaN,54
35,LogisticRegression,0.158014,0.341023,0.343137,0.791878,0.801005,"[cons.price.idx, cons.conf.idx]",59
32,LogisticRegression,0.157025,0.341570,0.342811,0.792375,0.801135,"[emp.var.rate, cons.conf.idx, nr.employed]",60
30,LogisticRegression,0.162767,0.341610,0.342577,0.792196,0.801052,"[emp.var.rate, cons.price.idx, cons.conf.idx]",60
31,LogisticRegression,0.158638,0.337943,0.341741,0.791074,0.801393,"[emp.var.rate, cons.price.idx, nr.employed]",59
29,LogisticRegression,0.154473,0.341287,0.341503,0.792137,0.801310,"[emp.var.rate, cons.price.idx, cons.conf.idx, nr.employed]",62
22,LogisticRegression,90.472476,0.369552,0.341110,0.800306,0.800167,NaN,373
25,LogisticRegression,0.151042,0.340655,0.340909,0.792483,0.801066,NaN,57
23,LogisticRegression,0.151504,0.341337,0.338762,0.792158,0.800931,NaN,52


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Socio-Economic Feature Binning – Logistic Regression Conclusions**

- Binning socio-economic features (`emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `nr.employed`) **does not improve validation performance** (val F1 ~0.345, AUROC ~0.801).  
- Using subsets of features slightly reduces performance (~0.341–0.343).  
- **Conclusion:** Keeping these features as numeric is sufficient; binning adds complexity without benefit, consistent with earlier statistical significance results.

</span>

#### Hyperparameter Tuning with GridSearchCV

In [22]:
from sklearn.model_selection import GridSearchCV

# create a pipeline based on previous experiments
pipe_lr_gs = build_pipeline(
    soceco_bin_cols=None,
    calendar_cols_mode="onehot",
    poly_degree=1,
    age_bin_mode="group",
    drop_cols=["duration", "loan", "housing", "euribor3m"],
    model=LogisticRegression(solver="liblinear", random_state=42)
)

# grid params to test
param_grid = {
    "classifier__C": [0.1, 0.5, 1, 2, 10, 20, 100],
    "classifier__l1_ratio": [0, 1],
}

# prepare grid_search
grid_search = GridSearchCV(
    estimator=pipe_lr_gs,
    param_grid=param_grid,
    scoring="f1",  # or "roc_auc"
    n_jobs=-1,     # use all CPUs
    cv=4,          # 4-fold cross-validation
    verbose=0
)

# map target to 1/0 for proper scoring="f1"
y_train_bin = y_train.map({"no": 0, "yes": 1})
y_val_bin = y_val.map({"no": 0, "yes": 1})

In [23]:
%%time
grid_search.fit(X_train, y_train_bin)

CPU times: user 568 ms, sys: 282 ms, total: 851 ms
Wall time: 10.4 s


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...liblinear'))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.1, 0.5, ...], 'classifier__l1_ratio': [0, 1]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",4
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displaye

In [24]:
# best found model evaluation
best_pipe = grid_search.best_estimator_
y_val_pred = best_pipe.predict(X_val)
y_val_prob = best_pipe.predict_proba(X_val)[:, 1]

val_f1 = f1_score(y_val_bin, y_val_pred)
val_auroc = roc_auc_score(y_val_bin, y_val_prob)

val_f1, val_auroc

(0.34549878345498786, 0.8011287383838861)

In [25]:
# run train_pipeline with best found hyperparams
best_pipe_lr = train_pipeline(X_train, X_val, y_train, y_val,
    {
        "soceco_bin_cols": None,
        "calendar_cols_mode": "onehot",
        "poly_degree": 1,
        "age_bin_mode": "group",
        "drop_cols": ["duration", "loan", "housing", "euribor3m"],
        "model": LogisticRegression(solver="liblinear", l1_ratio=0, C=10, random_state=42)
    }
)

In [26]:
show_results_df(
    show_cols=["model_name", "model__C", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc"],
    show_count=10,
)

,model_name,model__C,fit_time,train_f1,val_f1,train_auroc,val_auroc
36,LogisticRegression,10.0,0.158716,0.344030,0.345499,0.792247,0.801129
26,LogisticRegression,NaN,0.156620,0.342184,0.345219,0.792346,0.801223
24,LogisticRegression,NaN,0.155966,0.342184,0.345219,0.792346,0.801223
35,LogisticRegression,NaN,0.158014,0.341023,0.343137,0.791878,0.801005
32,LogisticRegression,NaN,0.157025,0.341570,0.342811,0.792375,0.801135
30,LogisticRegression,NaN,0.162767,0.341610,0.342577,0.792196,0.801052
31,LogisticRegression,NaN,0.158638,0.337943,0.341741,0.791074,0.801393
29,LogisticRegression,NaN,0.154473,0.341287,0.341503,0.792137,0.801310
22,LogisticRegression,NaN,90.472476,0.369552,0.341110,0.800306,0.800167
25,LogisticRegression,NaN,0.151042,0.340655,0.340909,0.792483,0.801066


<span style="background-color: #4FC3F7; display:block; padding:10px">

**GridSearchCV – Logistic Regression Conclusions**

- The best pipeline found via GridSearchCV achieves the **highest validation performance** among all experiments.  
- The optimal configuration uses **L2 regularization with C = 10**.  
- It is therefore reasonable to use this pipeline as the main **Logistic Regression model** for comparison with other models.

</span>

#### Save Logistic Regression Pipeline

In [27]:
# save to "models" folder
import joblib
from src.config import MODELS_DIR

joblib.dump(
    best_pipe_lr,
    MODELS_DIR / "log_reg_model_pipeline.joblib"
);

In [28]:
# smoke test if all had been saved properly and ca nbe used in future
pipe_lr_smoke = joblib.load(MODELS_DIR / "log_reg_model_pipeline.joblib")

y_val_pred = pipe_lr_smoke.predict(X_val)
y_val_prob = pipe_lr_smoke.predict_proba(X_val)[:, 1]

val_f1 = f1_score(y_val, y_val_pred, pos_label="yes")
val_auroc = roc_auc_score(y_val, y_val_prob)

val_f1, val_auroc

(0.34549878345498786, 0.8011287383838861)

### k-Nearest Neighbors (KNN)

Build a few KNN models to compare performance with other classifiers.

#### Drop Columns

Experiment with dropping additional features for the KNN model.

In [29]:
# use drop_cols_options created on previous (log reg) steps 
for drop_cols in drop_cols_options:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "drop_cols": drop_cols,
            "model": base_models[1],    # KNeighborsClassifier()
        }
    )

In [30]:
show_results_df(
    model_name="KNeighborsClassifier",
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__drop_cols", "n_features"],
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__drop_cols,n_features
41,KNeighborsClassifier,0.090924,0.488413,0.388889,0.899843,0.731071,"[duration, loan, housing, euribor3m, age]",51
45,KNeighborsClassifier,0.062778,0.461940,0.387187,0.768639,0.744909,"[duration, loan, housing, euribor3m, job, education, marital, age]",31
39,KNeighborsClassifier,0.081641,0.506527,0.385744,0.924910,0.743124,"[duration, loan, housing, euribor3m, job]",41
1,KNeighborsClassifier,0.094648,0.488967,0.383562,0.926449,0.746241,NaN,59
51,KNeighborsClassifier,0.053387,0.452936,0.381285,0.766054,0.741836,"[duration, loan, housing, euribor3m, job, education, marital, pdays, previous, age]",26
50,KNeighborsClassifier,0.052701,0.495494,0.379501,0.900241,0.747712,"[duration, loan, housing, euribor3m, job, education, marital, pdays, previous]",27
47,KNeighborsClassifier,0.083433,0.491352,0.377095,0.900718,0.733643,"[duration, loan, housing, euribor3m, pdays, age]",47
43,KNeighborsClassifier,0.071566,0.504254,0.375877,0.923960,0.746095,"[duration, loan, housing, euribor3m, job, marital]",38
49,KNeighborsClassifier,0.066783,0.479536,0.375436,0.855676,0.725625,"[duration, loan, housing, euribor3m, job, pdays, age, previous]",35
37,KNeighborsClassifier,0.107896,0.496787,0.372061,0.926663,0.741869,"[duration, loan, housing]",53


<span style="background-color: #4FC3F7; display:block; padding:10px">

**K-Nearest Neighbors (KNN) – Feature Dropping Experiments**

- KNN achieves **higher validation F1 (0.37–0.39) than Logistic Regression (~0.34)**, but training F1 is much higher, indicating **overfitting**.  
- Dropping features slightly reduces input dimensions and training time but **does not improve validation metrics**.  
- **Conclusion:** KNN can achieve better predictive performance on validation but is prone to overfitting, making it less stable than Logistic Regression.

</span>

#### Age / Socio-Economic Attributes Binning & Cyclical Encoding for Calendar Columns

Experiment with:

- Binning the numeric `age` feature into categorical groups to explore its effect on KNN model performance.  
- Binning selected socio-economic context features into categorical groups to reduce dimensionality and capture non-linear relationships.  
- Applying cyclical encoding to the `month` and `day_of_week` features to properly represent their periodic nature.

These preprocessing strategies aim to improve KNN performance by simplifying numeric features, capturing temporal patterns, and reducing noise from high-cardinality features.

In [31]:
for age_bin_mode in age_bin_mode_options:
    for calendar_cols_mode in calendar_cols_mode_options:
        for soceco_bin_cols in soceco_bin_cols_options:
            train_pipeline(X_train, X_val, y_train, y_val,
                {
                    "age_bin_mode": age_bin_mode,
                    "soceco_bin_cols": soceco_bin_cols,
                    "calendar_cols_mode": calendar_cols_mode,
                    "drop_cols": ["duration", "loan", "housing", "euribor3m"],
                    "model": base_models[1],
                }
            )

In [33]:
show_results_df(
    model_name="KNeighborsClassifier",
    show_cols=["model_name", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__age_bin_mode", "pipe__soceco_bin_cols", "pipe__calendar_cols_mode", "n_features"],
    show_count=20
)

,model_name,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__age_bin_mode,pipe__soceco_bin_cols,pipe__calendar_cols_mode,n_features
95,KNeighborsClassifier,0.117956,0.487327,0.403873,0.918721,0.735994,range,"[emp.var.rate, cons.price.idx, cons.conf.idx]",onehot,63
98,KNeighborsClassifier,0.116549,0.490684,0.402768,0.919530,0.743206,range,"[cons.price.idx, cons.conf.idx, nr.employed]",onehot,64
100,KNeighborsClassifier,0.099955,0.490744,0.397498,0.919264,0.743178,range,"[cons.price.idx, cons.conf.idx]",onehot,62
99,KNeighborsClassifier,0.100908,0.490152,0.395284,0.918849,0.743191,range,"[emp.var.rate, nr.employed]",onehot,60
106,KNeighborsClassifier,0.096844,0.503647,0.393855,0.919636,0.736706,range,"[emp.var.rate, nr.employed]",num,47
96,KNeighborsClassifier,0.110446,0.490533,0.393056,0.919239,0.740955,range,"[emp.var.rate, cons.price.idx, nr.employed]",onehot,62
94,KNeighborsClassifier,0.113423,0.487822,0.392184,0.919480,0.739739,range,"[emp.var.rate, cons.price.idx, cons.conf.idx, nr.employed]",onehot,65
61,KNeighborsClassifier,0.097013,0.498859,0.391825,0.926927,0.731054,None,"[emp.var.rate, cons.price.idx, nr.employed]",num,44
97,KNeighborsClassifier,0.104198,0.489261,0.391365,0.919609,0.740938,range,"[emp.var.rate, cons.conf.idx, nr.employed]",onehot,63
41,KNeighborsClassifier,0.090924,0.488413,0.388889,0.899843,0.731071,NaN,NaN,NaN,51


<span style="background-color: #4FC3F7; display:block; padding:10px">

**K-Nearest Neighbors (KNN) – Preprocessing Experiments**

- **Binning the numeric `age` feature** into **range categories** (e.g., 25–30, 31–40, etc.) **improves KNN validation performance**.  
- Applying **socio-economic feature binning** and **one-hot encoding for calendar features** further enhances validation F1 to **~0.40–0.403**, compared to ~0.37–0.39 without preprocessing.  
- Training F1 remains high (~0.49–0.50), showing KNN still **overfits**, but preprocessing reduces noise and slightly stabilizes validation performance.  
- The best combination uses **age range + binned 3 out of 4 socio-economic features + one-hot calendar encoding**, giving the highest validation F1 (~0.403).  
- **Conclusion:** Preprocessing is critical for KNN. Proper binning and encoding allow it to outperform Logistic Regression on validation F1, but overfitting persists and careful feature selection is necessary.

</span>

#### `n_neighbors` Hyperparameter Tuning

Experiment with different values of the `n_neighbors` hyperparameter to identify the optimal setting for the KNN model.

In [34]:
for n_neighbors in range(3, 9):
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "age_bin_mode": "range",
            "soceco_bin_cols": ["emp.var.rate", "cons.price.idx", "cons.conf.idx"],
            "calendar_cols_mode": "onehot",
            "drop_cols": ["duration", "loan", "housing", "euribor3m"],
            "model": KNeighborsClassifier(n_neighbors=n_neighbors)
        }
    )

In [40]:
show_results_df(
    model_name="KNeighborsClassifier",
    show_cols=["model_name", "model__n_neighbors", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc"],
    show_count=10
)

,model_name,model__n_neighbors,fit_time,train_f1,val_f1,train_auroc,val_auroc
117,KNeighborsClassifier,NaN,0.104928,0.487327,0.403873,0.918721,0.735994
95,KNeighborsClassifier,NaN,0.117956,0.487327,0.403873,0.918721,0.735994
98,KNeighborsClassifier,NaN,0.116549,0.490684,0.402768,0.919530,0.743206
100,KNeighborsClassifier,NaN,0.099955,0.490744,0.397498,0.919264,0.743178
99,KNeighborsClassifier,NaN,0.100908,0.490152,0.395284,0.918849,0.743191
106,KNeighborsClassifier,NaN,0.096844,0.503647,0.393855,0.919636,0.736706
115,KNeighborsClassifier,3.0,0.111653,0.571239,0.393548,0.929298,0.713742
96,KNeighborsClassifier,NaN,0.110446,0.490533,0.393056,0.919239,0.740955
94,KNeighborsClassifier,NaN,0.113423,0.487822,0.392184,0.919480,0.739739
61,KNeighborsClassifier,NaN,0.097013,0.498859,0.391825,0.926927,0.731054


<span style="background-color: #4FC3F7; display:block; padding:10px">

**KNN – `n_neighbors` Tuning Conclusions**

- The default value `n_neighbors = 5` appears to be optimal for the KNN model.

</span>

#### Save kNN Pipeline

In [41]:
pipeline_knn = build_pipeline(
    age_bin_mode="range",
    soceco_bin_cols=["emp.var.rate", "cons.price.idx", "cons.conf.idx"],
    calendar_cols_mode="onehot",
    drop_cols=["duration", "loan", "housing", "euribor3m"],
    model=KNeighborsClassifier(n_neighbors=5)
)

pipeline_knn.fit(X_train, y_train)
pipeline_knn

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feature_engineering', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols_initial', ...), ('campaign_prev', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['duration', 'loan', ...]"
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` f

In [42]:
joblib.dump(
    pipeline_knn,
    MODELS_DIR / "knn_model_pipeline.joblib"
);

In [43]:
import joblib
from src.config import MODELS_DIR

# smoke test if all had been saved properly and can be used in future
pipe_knn_smoke = joblib.load(MODELS_DIR / "knn_model_pipeline.joblib")

y_val_pred = pipe_knn_smoke.predict(X_val)
y_val_prob = pipe_knn_smoke.predict_proba(X_val)[:, 1]

val_f1 = f1_score(y_val, y_val_pred, pos_label="yes")
val_auroc = roc_auc_score(y_val, y_val_prob)

val_f1, val_auroc

(0.40387275242047027, 0.7359935167932449)

### Decision Tree

Build several single Decision Tree models to compare their performance with other classifiers.

#### Drop Columns

Experiment with dropping additional features to evaluate their impact on Decision Tree model performance.

In [44]:
# use drop_cols_options created on previous (log reg) steps
for drop_cols in drop_cols_options:
    for max_leaf_nodes in [100, 200, 400]: # experiment with few max_leaf_nodes options
        train_pipeline(X_train, X_val, y_train, y_val,
            {
                "drop_cols": drop_cols,
                "model": DecisionTreeClassifier(max_leaf_nodes=max_leaf_nodes, random_state=42),
            }
        )

In [45]:
# add keep all features experiments for decision tree
for max_leaf_nodes in [100, 200, 400]:
    train_pipeline(X_train, X_val, y_train, y_val,
        {
            "drop_cols": None,
            "model": DecisionTreeClassifier(max_leaf_nodes=max_leaf_nodes, random_state=42),
        }
    )

In [46]:
show_results_df(
    model_name="DecisionTreeClassifier",
    show_cols=["model_name", "model__max_leaf_nodes", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__drop_cols", "n_features"],
)

,model_name,model__max_leaf_nodes,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__drop_cols,n_features
122,DecisionTreeClassifier,200.0,0.133160,0.484274,0.417974,0.801277,0.777990,"[duration, loan, housing]",53
121,DecisionTreeClassifier,100.0,0.150632,0.448006,0.410779,0.794973,0.792376,"[duration, loan, housing]",53
167,DecisionTreeClassifier,200.0,0.177017,0.481146,0.410182,0.802291,0.775688,None,59
2,DecisionTreeClassifier,100.0,0.162604,0.452679,0.407924,0.795160,0.790662,NaN,59
166,DecisionTreeClassifier,100.0,0.153741,0.452679,0.407924,0.795160,0.790662,None,59
168,DecisionTreeClassifier,400.0,0.158310,0.539293,0.397380,0.814142,0.748748,None,59
159,DecisionTreeClassifier,400.0,0.097147,0.491731,0.390704,0.812955,0.770488,"[duration, loan, housing, euribor3m, job, pdays, age, previous]",35
129,DecisionTreeClassifier,400.0,0.116806,0.518572,0.389928,0.816518,0.750641,"[duration, loan, housing, euribor3m, job]",41
135,DecisionTreeClassifier,400.0,0.128801,0.509818,0.387050,0.815666,0.761966,"[duration, loan, housing, euribor3m, age]",51
141,DecisionTreeClassifier,400.0,0.106140,0.511347,0.386067,0.816250,0.760064,"[duration, loan, housing, euribor3m, job, marital]",38


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Decision Tree – Drop Columns & Max Leaf Nodes Experiments**

- **Max leaf nodes = 200** generally provides the **best validation F1 (~0.41–0.42)**; larger trees (400 leaves) overfit (high train F1, lower val F1).  
- For Decision Trees, **keeping all features except `loan` and `housing`** gives the best performance.  
- **Dropping additional features has little impact**, showing that tree-based models can **extract useful predictive information even from features with low statistical significance**.  
- **Conclusion:** Decision Trees outperform Logistic Regression (~0.34 F1) and slightly surpass KNN (~0.40 F1) when tuned, but careful control of tree size is needed to avoid overfitting.

</span>

#### Cyclical Encoding for Calendar Columns

Experiment with different encoding strategies for the `month` and `day_of_week` features for Decision tree model

In [52]:
for calendar_cols_mode in calendar_cols_mode_options:
    train_pipeline(X_train, X_val, y_train, y_val,
            {
                "calendar_cols_mode": calendar_cols_mode,
                "drop_cols": ["duration", "loan", "housing"],
                "model": DecisionTreeClassifier(max_leaf_nodes=200, random_state=42),
            }
        )

In [54]:
show_results_df(
    model_name="DecisionTreeClassifier",
    show_cols=["model_name", "model__max_leaf_nodes", "fit_time", "train_f1", "val_f1", "train_auroc", "val_auroc", "pipe__calendar_cols_mode", "n_features"],
    show_count=10
)

,model_name,model__max_leaf_nodes,fit_time,train_f1,val_f1,train_auroc,val_auroc,pipe__calendar_cols_mode,n_features
122,DecisionTreeClassifier,200.0,0.133160,0.484274,0.417974,0.801277,0.777990,NaN,53
172,DecisionTreeClassifier,200.0,0.171369,0.484274,0.417974,0.801277,0.777990,onehot,53
173,DecisionTreeClassifier,200.0,0.116263,0.465463,0.414045,0.797141,0.773552,num,40
121,DecisionTreeClassifier,100.0,0.150632,0.448006,0.410779,0.794973,0.792376,NaN,53
167,DecisionTreeClassifier,200.0,0.177017,0.481146,0.410182,0.802291,0.775688,NaN,59
2,DecisionTreeClassifier,100.0,0.162604,0.452679,0.407924,0.795160,0.790662,NaN,59
166,DecisionTreeClassifier,100.0,0.153741,0.452679,0.407924,0.795160,0.790662,NaN,59
174,DecisionTreeClassifier,200.0,0.122598,0.459010,0.407653,0.797212,0.759464,cyclical,42
168,DecisionTreeClassifier,400.0,0.158310,0.539293,0.397380,0.814142,0.748748,NaN,59
159,DecisionTreeClassifier,400.0,0.097147,0.491731,0.390704,0.812955,0.770488,NaN,35


<span style="background-color: #4FC3F7; display:block; padding:10px">

**Decision Tree – Calendar Columns Encoding Experiments**

- Changing encoding for `month` and `day_of_week` has **minimal impact** on validation F1 (~0.414–0.418) for Decision Trees.  
- **One-hot, numeric, or cyclical encodings** all give similar results, confirming that **Decision Trees handle categorical/time features naturally**.  
- **Conclusion:** Calendar feature encoding is not critical for tree-based models; keeping the features as-is or using simple encodings suffices.

</span>

#### Hyperparameter Tuning with RandomizedSearchCV

tune hyperparams for DecisionTreeClassifier model 

In [65]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "classifier__max_leaf_nodes": [50, 100, 200, 400, 600, None],
    "classifier__min_samples_split": [2, 5, 10, 20],
    "classifier__min_samples_leaf": [1, 2, 5, 10],
    "classifier__max_depth": range(5, 25),
    "classifier__criterion": ["gini", "entropy"]
}

dt_pipeline = build_pipeline(
    drop_cols=["duration", "loan", "housing"],
    model=DecisionTreeClassifier(random_state=42)
)

n_iter_search = 200  # number of random parameter sets to try
dt_random_search = RandomizedSearchCV(
    dt_pipeline,
    param_distributions=param_dist,
    n_iter=n_iter_search,
    scoring='f1',        # optimize for validation F1
    cv=4,                # 4-fold cross-validation
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# map target to 1/0 for proper scoring="f1"
y_train_bin = y_train.map({"no": 0, "yes": 1})
y_val_bin = y_val.map({"no": 0, "yes": 1})

In [66]:
dt_random_search.fit(X_train, y_train_bin)

Fitting 4 folds for each of 200 candidates, totalling 800 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__criterion': ['gini', 'entropy'], 'classifier__max_depth': range(5, 25), 'classifier__max_leaf_nodes': [50, 100, ...], 'classifier__min_samples_leaf': [1, 2, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",200
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscros

In [67]:
best_dt = dt_random_search.best_estimator_
y_val_pred = best_dt.predict(X_val)
y_val_prob = best_dt.predict_proba(X_val)[:, 1]

from sklearn.metrics import f1_score, roc_auc_score

val_f1 = f1_score(y_val_bin, y_val_pred)
val_auroc = roc_auc_score(y_val_bin, y_val_prob)

print("Best Decision Tree parameters:", dt_random_search.best_params_)
print("Validation F1:", val_f1)
print("Validation AUROC:", val_auroc)

Best Decision Tree parameters: {'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 5, 'classifier__max_leaf_nodes': 100, 'classifier__max_depth': 22, 'classifier__criterion': 'entropy'}
Validation F1: 0.4065281899109792
Validation AUROC: 0.802554734303505


#### Save Decision Tree Pipeline

In [69]:
pipeline_dt = train_pipeline(X_train, X_val, y_train, y_val,
    {
        "drop_cols": ["duration", "loan", "housing"],
        "model": DecisionTreeClassifier(
            min_samples_split=10,
            min_samples_leaf=5,
            max_leaf_nodes=100,
            max_depth=22,
            criterion="entropy",
            random_state=42,
        )
    }
)
pipeline_dt

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feature_engineering', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols_initial', ...), ('campaign_prev', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns,"['duration', 'loan', ...]"
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` f

In [72]:
joblib.dump(
    pipeline_dt,
    MODELS_DIR / "decision_tree_model_pipeline.joblib"
);

In [73]:
# smoke test if all had been saved properly and can be used in future
pipe_dt_smoke = joblib.load(MODELS_DIR / "decision_tree_model_pipeline.joblib")

y_val_pred = pipe_dt_smoke.predict(X_val)
y_val_prob = pipe_dt_smoke.predict_proba(X_val)[:, 1]

val_f1 = f1_score(y_val, y_val_pred, pos_label="yes")
val_auroc = roc_auc_score(y_val, y_val_prob)

val_f1, val_auroc

(0.4065281899109792, 0.802554734303505)